In [ ]:
# Setup.
import os
import sys
from pathlib import Path

for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import matplotlib.pyplot as plt  # noqa: E402
from matplotlib.cm import ScalarMappable  # noqa: E402
from matplotlib.colors import Normalize  # noqa: E402
from matplotlib.lines import Line2D  # noqa: E402
from matplotlib.transforms import Bbox, blended_transform_factory  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

from scripts.run_corrca_snr_grid import corrca_operators  # noqa: E402
from src.analysis import iva_quality  # noqa: E402
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    CoordinateSystems,
    ExperimentNames,
    IvaVariants,
    MusicTypeVariants,
    SpectrumTypeVariants,
)
from src.io.iva_store import load_iva_components  # noqa: E402
from src.io.loading import (  # noqa: E402
    assr_electrode_mask,
    load_snr_courses,
    load_snr_values,
)
from src.visualization.assr_filter_plots import (  # noqa: E402
    plot_masked_filter_topomaps,
)
from src.visualization.assr_snr_plots import (  # noqa: E402
    CONDITION_COLORS,
    DIVERGING_CMAP,
    plot_condition_tf_grid,
    plot_courses,
)
from src.visualization.corrca_plots import (  # noqa: E402
    COMPONENT_PREFIX,
    plot_shared_topomaps,
)

# The imported scripts switch matplotlib to the headless backend; switch back.
%matplotlib inline

print("Setup complete.")

# Conference poster: result figures

The data figures of the poster: the TF maps, the topomaps, the 40 Hz SNR trial courses
and the condition bar plots. Same data, same checks and same drawing functions as the
result sections of [`poster_figures.ipynb`](poster_figures.ipynb), which describes each
read-out in full; this notebook only restyles them for the poster:

- larger tick numbers (`TICK_FONTSIZE`);
- no titles or axis labels — they are added on the poster by hand, and every cell
  prints what it left out (colour key, scale limits); the only text kept is the SNR
  plots' method legend and the bar plots' p-values;
- each figure saved as a PDF and an SVG, raster parts (TF maps, topomaps) at `DPI`, the
  text kept as editable text in both.

Everything goes under `plots/conference/results/<figure>/`. Methods:
[`docs/analyses/06b-corrca-condition-comparison/`](../../docs/analyses/06b-corrca-condition-comparison/README.md)
and [`docs/analyses/00-preprocessing/`](../../docs/analyses/00-preprocessing/README.md).

In [ ]:
TICK_FONTSIZE = 14
DPI = 600
FORMATS = ("pdf", "svg")
TRANSPARENT = True  # transparent background, for placing on a coloured poster
LINE_COLOR = "#1f2937"
COMPONENT = 1  # 1-based: IC1 / CC1 / PC1
CONDITIONS = (ConditionVariants.PLACEBO.value, ConditionVariants.PSILOCYBIN.value)

# Text stays text (not paths) in both formats, so it can be edited on the poster.
plt.rcParams.update({"pdf.fonttype": 42, "svg.fonttype": "none"})

CORRCA_VARIANT = IvaVariants.CORRCA_CHANNEL_JOINED_TRACKS
CORRCA_N_PCA = 5
STAGE00_PIPELINE = "2_roi_zscore_pca"
STAGE00_N_COMPONENTS = 3  # the stage-00 job's --n_components

_RESULTS = ProjectPaths.PROJECT_ROOT / "results" / ExperimentNames.ASSR.value
_STAGE00_RESULTS = _RESULTS / "assr_roi_tf_variants" / MusicTypeVariants.ASSR.value
_CORRCA_GRID = (
    f"assr_snr_grid__{CORRCA_VARIANT.value}__{SpectrumTypeVariants.BROADBAND.value}"
    f"__pca{CORRCA_N_PCA}"
)


def output_dir(figure: str) -> Path:
    path = ProjectPaths.analysis_plots_dir(
        "conference", "results", analysis_type=figure
    )
    path.mkdir(parents=True, exist_ok=True)
    return path


def save(fig: plt.Figure, stem: Path, bbox: Bbox | str = "tight") -> None:
    """Save *fig* as ``<stem>.pdf`` and ``<stem>.svg``."""
    # Not with_suffix: stems such as "...__0-0.5s" hold a dot of their own.
    for fmt in FORMATS:
        fig.savefig(
            stem.parent / f"{stem.name}.{fmt}",
            dpi=DPI,
            bbox_inches=bbox,
            transparent=TRANSPARENT,
        )


def strip_text(fig: plt.Figure, labelsize: float = TICK_FONTSIZE) -> None:
    """Drop every title, axis label and legend of *fig*; set its tick numbers to
    *labelsize*."""
    fig.suptitle("")
    for legend in fig.legends:
        legend.remove()
    for ax in fig.axes:
        for loc in ("left", "center", "right"):
            ax.set_title("", loc=loc)
        ax.set_xlabel("")
        ax.set_ylabel("")
        if ax.get_legend() is not None:
            ax.get_legend().remove()
        ax.tick_params(labelsize=labelsize)

## TF maps

2 × 2 grids of trial-normalised TF maps (each trial referenced to its own pre-stimulus
window, in baseline SD), Placebo left, Psilocybin right, one figure per entry of
`TF_FIGURES`:

- **ROI PCA vs CorrCA**: stage-00 ROI → z-score → PCA PC1 on top, CorrCA CC1 (the
  full-montage fit) below;
- **CorrCA, full montage vs ROI fit**: CorrCA CC1 of the full-montage fit on top, and
  below CC1 of the stage-06b fit on the ASSR ROI alone (`_roi` store: the channels cut
  to the ROI before the z-scoring, then PCA → CorrCA; see
  [§5b](../../docs/analyses/06b-corrca-condition-comparison/README.md)).

Each row is drawn by the jobs' `plot_condition_tf_grid`, recoloured in
`DIVERGING_CMAP` and copied into the grid unchanged. The four maps of a figure share
one symmetric scale, shown by a single colour bar right of the grid:
`TF_COLORBAR_LENGTH` of the two rows' height, centred on them. Tick numbers on the
outer edges only.

In [ ]:
TF_PANEL = (3.0, 2.4)  # inches per panel
TF_SPACING = 0.08  # gap between panels, as a fraction of a panel
TF_GUIDE_COLOR = LINE_COLOR  # onset/offset and 40 Hz lines
TF_COLORBAR_LENGTH = 0.4  # fraction of the two rows' height
TF_COLORBAR_WIDTH = 0.1  # inches
TF_COLORBAR_GAP = 0.12  # inches between the grid and the colour bar
TF_COLORBAR_FONTSIZE = 11

CORRCA_ROI_VARIANT = IvaVariants.CORRCA_CHANNEL_JOINED_TRACKS_ROI
CORRCA_TF_PATH = _RESULTS / f"{_CORRCA_GRID}__tf_baseline.npz"
CORRCA_ROI_TF_PATH = _RESULTS / (
    f"assr_snr_grid__{CORRCA_ROI_VARIANT.value}__"
    f"{SpectrumTypeVariants.BROADBAND.value}__pca{CORRCA_N_PCA}__tf_baseline.npz"
)
STAGE00_TF_PATH = _STAGE00_RESULTS / f"tf_maps__pca{STAGE00_N_COMPONENTS}.npz"

# Output stem -> its rows, top to bottom: row label -> (TF store, its key per
# condition). Columns are CONDITIONS.
_CORRCA_ROW = (CORRCA_TF_PATH, "tf_baseline__{}")
TF_FIGURES = {
    f"tf_grid_roi_pc{COMPONENT}_corrca_cc{COMPONENT}": {
        f"{STAGE00_PIPELINE} PC{COMPONENT}": (
            STAGE00_TF_PATH,
            f"tf_baseline__{STAGE00_PIPELINE}__{{}}",
        ),
        f"CorrCA CC{COMPONENT}": _CORRCA_ROW,
    },
    f"tf_grid_corrca_cc{COMPONENT}_full_vs_roi_fit": {
        f"CorrCA CC{COMPONENT}, full montage": _CORRCA_ROW,
        f"CorrCA CC{COMPONENT}, ROI fit": (CORRCA_ROI_TF_PATH, "tf_baseline__{}"),
    },
}

In [ ]:
def restyle_tf(fig: plt.Figure) -> None:
    """Redraw every TF map of *fig* in ``DIVERGING_CMAP`` and its guide lines in
    ``TF_GUIDE_COLOR``; data, colour limits and line styles are unchanged."""
    for ax in fig.axes:
        if not ax.images:
            continue
        ax.images[0].set_cmap(DIVERGING_CMAP)
        for line in ax.lines:
            line.set_color(TF_GUIDE_COLOR)


def copy_tf_panel(src: plt.Axes, dst: plt.Axes) -> None:
    """Redraw the TF image and guide lines of *src* on *dst*, unchanged."""
    image = src.images[0]
    dst.imshow(
        image.get_array(),
        aspect="auto",
        origin=image.origin,
        extent=image.get_extent(),
        cmap=image.cmap,
        norm=image.norm,
        interpolation=image.get_interpolation(),
    )
    # axvline / axhline are drawn in blended data-axes coordinates; keep each line in
    # the matching coordinates of the new axes.
    transforms = {
        id(src.get_xaxis_transform()): dst.get_xaxis_transform(),
        id(src.get_yaxis_transform()): dst.get_yaxis_transform(),
    }
    for line in src.lines:
        dst.add_line(
            Line2D(
                *line.get_data(),
                color=line.get_color(),
                linestyle=line.get_linestyle(),
                linewidth=line.get_linewidth(),
                transform=transforms.get(id(line.get_transform()), dst.transData),
            )
        )
    dst.set_xlim(src.get_xlim())
    dst.set_ylim(src.get_ylim())


def add_side_colorbar(fig: plt.Figure, axes: np.ndarray) -> plt.Axes:
    """One vertical colour bar for the image of ``axes[0, 0]``, right of the grid and
    centred on its height."""
    right = axes[0, -1].get_position().x1
    bottom = axes[-1, 0].get_position().y0
    top = axes[0, 0].get_position().y1
    length = TF_COLORBAR_LENGTH * (top - bottom)
    gap = TF_COLORBAR_GAP / fig.get_figwidth()
    width = TF_COLORBAR_WIDTH / fig.get_figwidth()
    cax = fig.add_axes([right + gap, (top + bottom - length) / 2, width, length])
    fig.colorbar(axes[0, 0].images[0], cax=cax)
    cax.tick_params(labelsize=TF_COLORBAR_FONTSIZE)
    return cax

In [ ]:
def plot_tf_grid(
    rows: dict[str, tuple[Path, str]],
) -> tuple[plt.Figure, float, int]:
    """One row per entry of *rows*, one column per condition, every map on one
    symmetric scale and a single colour bar; returns the figure, that scale's limit
    and the cohort size. The stores must share their axes and their participants."""
    stores = {row: np.load(path) for row, (path, _) in rows.items()}
    first = next(iter(stores.values()))
    epoch_times, freqs = first["epoch_times"], first["freqs"]
    for row, store in stores.items():
        if not (
            np.allclose(store["epoch_times"], epoch_times)
            and np.allclose(store["freqs"], freqs)
        ):
            raise ValueError(f"{row}: TF maps on a different time/frequency axis.")
    cohorts = {
        row: sorted(str(p) for p in store["participants"])
        for row, store in stores.items()
    }
    if len({tuple(cohort) for cohort in cohorts.values()}) > 1:
        raise ValueError(f"The rows' cohorts differ: {cohorts}")

    k = COMPONENT - 1
    # Row -> condition -> (1, F, W): the mean over participants of each participant's
    # trial-median map.
    group = {
        row: {c: stores[row][key.format(c)][k : k + 1].mean(axis=1) for c in CONDITIONS}
        for row, (_, key) in rows.items()
    }
    shared = max(
        float(np.nanmax(np.abs(m))) for by_c in group.values() for m in by_c.values()
    )

    width, height = TF_PANEL
    fig, axes = plt.subplots(
        len(rows),
        len(CONDITIONS),
        figsize=(width * len(CONDITIONS), height * len(rows)),
        gridspec_kw={"wspace": TF_SPACING, "hspace": TF_SPACING},
        squeeze=False,
    )
    for r, row in enumerate(rows):
        job_fig = plot_condition_tf_grid(
            group[row],
            {c: [""] for c in CONDITIONS},
            list(CONDITIONS),
            epoch_times,
            freqs,
            iva_quality.ASSR_FREQ,
            "",
            None,
            show_difference=False,
            close=False,
        )
        restyle_tf(job_fig)
        # The TF axes are the ones holding an image; the rest are their colour bars.
        job_axes = [ax for ax in job_fig.axes if ax.images]
        for col, job_ax in enumerate(job_axes):
            job_ax.images[0].set_clim(-shared, shared)
            copy_tf_panel(job_ax, axes[r, col])
            axes[r, col].tick_params(
                labelsize=TICK_FONTSIZE,
                labelleft=col == 0,
                labelbottom=r == len(rows) - 1,
            )
        plt.close(job_fig)
    add_side_colorbar(fig, axes)
    return fig, shared, len(first["participants"])

In [ ]:
tf_dir = output_dir("tf_maps")
for stem, rows in TF_FIGURES.items():
    fig, shared, n = plot_tf_grid(rows)
    save(fig, tf_dir / stem)
    plt.show()
    print(f"{stem}: rows {' / '.join(rows)} (top to bottom)")
    print(f"  columns {', '.join(CONDITIONS)}; x: time (s), y: frequency (Hz)")
    print(f"  colour bar ±{shared:.3g} baseline SD (n={n}); saved to {tf_dir}")

## Topomaps: CorrCA CC1 and the binary ASSR ROI

The shared forward pattern of CorrCA CC1 (stage-06b JoinedTracks fit) on its symmetric
scale, and the ASSR ROI electrodes as a 0/1 map on the same channels — both through the
job's `plot_shared_topomaps`, in `DIVERGING_CMAP`. The ROI is drawn with
nearest-electrode interpolation and no contours so it stays two-valued. CC1 has a bare
colour bar of its own, relative (−1 to 1); the cell prints its absolute limit.

In [ ]:
TOPO_COORDINATES = CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS
TOPO_MASK_INTERP = "nearest"
TOPO_MASK_CONTOURS = 0
TOPO_COLORBAR_SIZE = (0.3, 2.6)  # inches

In [ ]:
def save_relative_colorbar(ax: plt.Axes, stem: Path) -> float:
    """Save a bare colour bar for the image on *ax*, relative to its symmetric limit
    (-1 to 1); return the limit."""
    image = ax.images[0]
    fig, cax = plt.subplots(figsize=TOPO_COLORBAR_SIZE)
    # A mappable of its own: the image belongs to another figure.
    fig.colorbar(
        ScalarMappable(norm=Normalize(-1.0, 1.0), cmap=image.cmap),
        cax=cax,
        ticks=[-1.0, -0.5, 0.0, 0.5, 1.0],
    )
    cax.tick_params(labelsize=TICK_FONTSIZE)
    save(fig, stem)
    plt.close(fig)
    return float(image.get_clim()[1])

In [ ]:
topo_dir = output_dir("topomaps")
corrca_topo = load_iva_components(
    experiment=ExperimentNames.ASSR,
    condition=ConditionVariants.JOINED_TRACKS,
    variant=CORRCA_VARIANT,
    music_type=MusicTypeVariants.ASSR,
    n_pca=CORRCA_N_PCA,
)
topo_info = corrca_topo.topo_info(TOPO_COORDINATES)
k = COMPONENT - 1
roi_mask = assr_electrode_mask(
    list(corrca_topo.channel_names), TOPO_COORDINATES, strict=True
)
# Output stem -> (1, C) pattern, (1,) ISC for the job's title (stripped), and the
# drawing options; CC1 keeps the job's own.
topomaps = {
    f"corrca_cc{COMPONENT}": (
        corrca_topo.extras["corrca_shared_patterns"][k : k + 1],
        corrca_topo.extras["corrca_fit_isc"][k : k + 1],
        {},
    ),
    "assr_roi_mask": (
        roi_mask[None].astype(float),
        np.full(1, np.nan),
        {"image_interp": TOPO_MASK_INTERP, "contours": TOPO_MASK_CONTOURS},
    ),
}
for stem, (pattern, isc, options) in topomaps.items():
    fig = plot_shared_topomaps(
        pattern,
        isc,
        topo_info,
        corrca_topo.n_channels,
        [0],
        label=stem,
        cmap=DIVERGING_CMAP,
        **options,
    )
    strip_text(fig)
    save(fig, topo_dir / stem)
    plt.show()
    if stem.startswith("corrca"):
        vlim = save_relative_colorbar(fig.axes[0], topo_dir / f"{stem}_colorbar")
        print(f"{stem}: colour bar -1..1 = ±{vlim:.3g}; saved")
    else:
        print(f"{stem}: {int(roi_mask.sum())} of {roi_mask.size} electrodes; saved")

## 40 Hz SNR over the trial: classical approach vs CorrCA

One file per condition (Placebo, Psilocybin), each overlaying the two methods' 40 Hz
course (mean ± SEM over participants, in baseline SD): the classical stage-00 ROI PCA
PC1 and our CorrCA CC1, drawn by the jobs' `plot_courses`, as in
`poster_figures.ipynb`. Both files share one y-range. Stimulus shaded grey,
`COURSE_WINDOW` gold. Titles and axis names are stripped; each file keeps a method
legend (`COURSE_LEGEND_*`), and tick numbers are `COURSE_TICK_FONTSIZE`.

In [ ]:
COURSE_SELECTION = "40hz"  # or "35-45hz"
COURSE_WINDOW = (0.0, 0.5)  # shaded gold
COURSE_CLASSICAL = "ROI PCA PC1"  # or "ROI", the plain electrode average
COURSE_TICK_FONTSIZE = 18
COURSE_LEGEND_FONTSIZE = 14
COURSE_LEGEND_GAP = 0.02  # s between the shaded stimulus/window and the legend
COURSE_FIGSIZE = (5.1, 2.6)  # inches; the job draws a panel 5.1 x 3.6

_STAGE00_COURSES = (
    _STAGE00_RESULTS / f"snr_courses__full__pca{STAGE00_N_COMPONENTS}.npz"
)
# Method label -> the courses file, the run within it, and the source within that run.
COURSE_SOURCES = {
    "ROI PCA PC1": (_STAGE00_COURSES, STAGE00_PIPELINE, f"PC {COMPONENT}"),
    "ROI": (_STAGE00_COURSES, STAGE00_PIPELINE, "ASSR-mask (full)"),
    f"CorrCA {COMPONENT_PREFIX}{COMPONENT}": (
        _RESULTS / f"{_CORRCA_GRID}__courses.npz",
        "full",
        f"{COMPONENT_PREFIX} {COMPONENT}",
    ),
}
# Classical first, then ours; near-black and the Okabe-Ito bluish green, which stay
# apart under every colour-vision deficiency and differ from the condition colours.
COURSE_METHODS = (COURSE_CLASSICAL, f"CorrCA {COMPONENT_PREFIX}{COMPONENT}")
COURSE_COLORS = dict(zip(COURSE_METHODS, (LINE_COLOR, "#009E73")))

In [ ]:
loaded = {m: load_snr_courses(COURSE_SOURCES[m][0]) for m in COURSE_METHODS}
cohorts = {m: sorted(item[0], key=int) for m, item in loaded.items()}
participants = cohorts[COURSE_METHODS[0]]
if any(cohort != participants for cohort in cohorts.values()):
    raise ValueError(f"The methods' cohorts differ: {cohorts}")
times = loaded[COURSE_METHODS[0]][2]
if any(not np.allclose(item[2], times) for item in loaded.values()):
    raise ValueError("The methods' epoch time axes differ.")

# Method -> condition -> (participants, times), on the shared participant order.
by_method = {}
for m, (run_participants, sources, _, courses) in loaded.items():
    _, run, source = COURSE_SOURCES[m]
    order = [run_participants.index(p) for p in participants]
    by_condition = courses[(run, COURSE_SELECTION)]
    by_method[m] = {
        c: by_condition[c][order, sources.index(source)] for c in CONDITIONS
    }

figures = {}
for c in CONDITIONS:
    # The condition takes the place of plot_courses's one source, and the methods
    # that of its conditions: method -> (participants, 1, times).
    figures[c] = plot_courses(
        {m: by_method[m][c][:, None] for m in COURSE_METHODS},
        times,
        COURSE_WINDOW,
        None,
        [c],
        list(COURSE_METHODS),
        0.05,
        "",
        None,
        close=False,
        colors=COURSE_COLORS,
    )
    strip_text(figures[c], COURSE_TICK_FONTSIZE)
    figures[c].set_size_inches(COURSE_FIGSIZE)
    # Top left corner just past the shaded stimulus and window: x in seconds, y at
    # the top of the axes.
    ax = figures[c].axes[0]
    shaded_end = max(COURSE_WINDOW[1], AssrEpoch.STIMULUS_DURATION_S)
    ax.legend(
        loc="upper left",
        bbox_to_anchor=(shaded_end + COURSE_LEGEND_GAP, 1.0),
        bbox_transform=blended_transform_factory(ax.transData, ax.transAxes),
        borderaxespad=0.2,
        frameon=False,
        fontsize=COURSE_LEGEND_FONTSIZE,
    )

# One y-range over the conditions, so their files compare directly.
limits = [fig.axes[0].get_ylim() for fig in figures.values()]
ylim = (min(low for low, _ in limits), max(high for _, high in limits))
course_dir = output_dir("trial_courses")
lo, hi = COURSE_WINDOW
slug = COURSE_CLASSICAL.lower().replace(" ", "_")
for c, fig in figures.items():
    fig.axes[0].set_ylim(ylim)
    fig.tight_layout()
    save(
        fig,
        course_dir
        / (
            f"{slug}_vs_corrca_cc{COMPONENT}_{c.lower()}__{COURSE_SELECTION}__"
            f"{lo:g}-{hi:g}s"
        ),
    )
plt.show()
print(f"files: {', '.join(CONDITIONS)}; x: time from onset (s), y: SNR (baseline SD)")
print(f"colours: {', '.join(f'{m} {c}' for m, c in COURSE_COLORS.items())}")
print(f"saved to {course_dir} (n={len(participants)})")

## Condition bar plots: four methods

The values the Placebo − Psilocybin tests run on: per participant and condition, the
median over trials of the 40 Hz SNR averaged over `STATS_WINDOW`, for the first
component of ICA (stage 07), IVA (stage 06), CorrCA (stage 06b) and the baseline ROI PCA
(stage 00). Two families of figures, **one file per method**:

- `condition_bars/`: Placebo and Psilocybin bars (mean ± SEM), each participant's pair
  joined in grey;
- `condition_delta/`: one Psilocybin − Placebo bar (mean ± SEM), a dot per participant;
  the learned methods in green, the baseline in near-black.

Each file carries the job's p-value (exact Wilcoxon signed-rank on the paired
differences, one-sided Placebo > Psilocybin, uncorrected), read from its CSV: over a
bracket joining the two bars, or over the single delta bar, just above that method's
own highest point. Over a bracket the text is `STATS_P_FONTSIZE` and may overhang it;
over a delta bar it is shrunk to be no wider than the bar. Only
`STATS_Y_AXIS` (ICA) keeps its y-axis; every other file is the bars, the p-value and
the zero line across the bars, nothing else. Within a family all files share one
y-range, every bar is equally wide, and every file is cropped to the same height (the
axes box plus half a tick label above and below), so they line up side by side with
the ICA one at the left. The baseline's bars are hatched. The cell checks each
values file against its job's CSV, so the bars are exactly what was tested.

In [ ]:
STATS_SELECTION = "40hz"  # or "35-45hz"
STATS_WINDOW = (0.0, 0.5)  # s
STATS_Y_AXIS = f"ICA IC{COMPONENT}"  # the one method drawn with its y-axis
STATS_SHOW_PARTICIPANTS = True  # False: bars and error bars only
STATS_X_UNIT = 1.4  # inches per bar slot, so every bar is equally wide
STATS_HEIGHT = 3.0  # inches of axes
STATS_BAR_WIDTH = 0.8  # of a slot
STATS_ZERO_MARGIN = 0.1  # how far the zero line runs past the outer bars, in slots
STATS_ZERO_WIDTH = 1.2  # points
STATS_P_FONTSIZE = 16  # over a delta bar, shrunk until no wider than the bar
STATS_P_GAP = 0.04  # bracket above the method's highest point, as a fraction of range
STATS_OURS_COLOR = "#009E73"  # delta bars of the learned methods
STATS_BASELINE_COLOR = LINE_COLOR  # delta bar of the baseline
STATS_BASELINE_HATCH = "///"
STATS_PAIR_COLOR = "#9ca3af"  # participant lines and dots
STATS_IVA_N_PCA = 5
STATS_JICA_N_ICA = 5

_lo, _hi = STATS_WINDOW
_GRID_WINDOW = f"{_lo:g}-{_hi:g}s"  # stage 00 / 06 / 06b
# run_wavelet_jica.py labels its one window by how it was set.
_JICA_WINDOW = (
    f"0-{_hi:.3f} s (paradigm default)"
    if STATS_WINDOW == (0.0, AssrEpoch.STIMULUS_DURATION_S)
    else f"{_lo:.3f}-{_hi:.3f} s (custom)"
)
_IVA_GRID = (
    f"assr_snr_grid__iva_channel_joined_tracks__"
    f"{SpectrumTypeVariants.BROADBAND.value}__pca{STATS_IVA_N_PCA}"
)
_JICA_TESTS = f"jica_tests__joined_tracks__ASSR__ica{STATS_JICA_N_ICA}"
_STAGE00_GRID = f"full__pca{STAGE00_N_COMPONENTS}"

# Method -> its test CSV and values file, the CSV column naming the run and the run,
# the window as that run labels it, the source, and whether it is the baseline. The
# order is the poster's left to right.
STATS_METHODS = {
    f"ICA IC{COMPONENT}": {
        "csv": _RESULTS / f"{_JICA_TESTS}.csv",
        "values": _RESULTS / f"{_JICA_TESTS}__values.npz",
        "run_column": "variant",
        "run": "full",
        "window": _JICA_WINDOW,
        "source": f"IC {COMPONENT}",
        "baseline": False,
    },
    f"IVA IC{COMPONENT}": {
        "csv": _RESULTS / f"{_IVA_GRID}.csv",
        "values": _RESULTS / f"{_IVA_GRID}__values.npz",
        "run_column": "variant",
        "run": "full",
        "window": _GRID_WINDOW,
        "source": f"IC {COMPONENT}",
        "baseline": False,
    },
    f"CorrCA {COMPONENT_PREFIX}{COMPONENT}": {
        "csv": _RESULTS / f"{_CORRCA_GRID}.csv",
        "values": _RESULTS / f"{_CORRCA_GRID}__values.npz",
        "run_column": "variant",
        "run": "full",
        "window": _GRID_WINDOW,
        "source": f"{COMPONENT_PREFIX} {COMPONENT}",
        "baseline": False,
    },
    f"ROI PCA PC{COMPONENT}": {
        "csv": _STAGE00_RESULTS / f"snr_grid__{_STAGE00_GRID}.csv",
        "values": _STAGE00_RESULTS / f"snr_values__{_STAGE00_GRID}.npz",
        "run_column": "pipeline",
        "run": STAGE00_PIPELINE,
        "window": _GRID_WINDOW,
        "source": f"PC {COMPONENT}",
        "baseline": True,
    },
}

In [ ]:
def method_values(label: str) -> tuple[dict[str, np.ndarray], pd.Series]:
    """One method's per-participant values (condition -> ``(P,)``) and the job's
    Placebo - Psilocybin test row; checks the values reproduce the test's median."""
    method = STATS_METHODS[label]
    _, sources, values = load_snr_values(method["values"])
    cell = values[(method["run"], STATS_SELECTION, method["window"])]
    column = sources.index(method["source"])
    by_condition = {c: cell[c][:, column] for c in CONDITIONS}

    frame = pd.read_csv(method["csv"])
    test = frame[
        (frame[method["run_column"]] == method["run"])
        & (frame["selection"] == STATS_SELECTION)
        & (frame["window"] == method["window"])
        & (frame["family"] == "contrast")
        & (frame["source"] == method["source"])
    ]
    if len(test) != 1:
        raise ValueError(
            f"{method['csv'].name}: {len(test)} contrast rows for {label}."
        )
    test = test.iloc[0]
    first, second = CONDITIONS
    median = np.nanmedian(by_condition[first] - by_condition[second])
    if not np.isclose(median, test["median"], rtol=1e-4, atol=1e-6):
        raise ValueError(
            f"{label}: the values give median {median:.4g}, the CSV {test['median']:.4g}"
            " — the two files are from different runs."
        )
    return by_condition, test


def _mean_sem(values: np.ndarray) -> tuple[float, float]:
    finite = values[np.isfinite(values)]
    return float(finite.mean()), float(finite.std(ddof=1) / np.sqrt(finite.size))


def _hatch(label: str) -> str | None:
    return STATS_BASELINE_HATCH if STATS_METHODS[label]["baseline"] else None


def _extent(groups: list[np.ndarray]) -> tuple[float, float]:
    """Lowest and highest point drawn for *groups*: the participants' values, or the
    bars' mean -/+ SEM without them."""
    if STATS_SHOW_PARTICIPANTS:
        every = np.concatenate(groups)
        return float(np.nanmin(every)), float(np.nanmax(every))
    ends = [m + s * sign for m, s in map(_mean_sem, groups) for sign in (-1, 1)]
    return min(ends), max(ends)


def bar_layout(
    extents: dict[str, tuple[float, float]],
) -> tuple[tuple[float, float], float, dict[str, float]]:
    """One y-range over every method's extent (always holding zero), the top of the
    y-axis, and each method's p-value bracket, just above its own highest point; the
    range leaves room above the highest bracket for the text."""
    low = min(0.0, *(lo for lo, _ in extents.values()))
    high = max(0.0, *(hi for _, hi in extents.values()))
    gap = STATS_P_GAP * (high - low)
    brackets = {label: max(hi, 0.0) + gap for label, (_, hi) in extents.items()}
    bottom = low - 0.05 * (high - low)
    axis_top = high + gap
    # The text takes a fixed share of the axes height, which itself depends on the
    # top: top - axis_top = share * (top - bottom), solved for top.
    share = 1.6 * STATS_P_FONTSIZE / 72 / STATS_HEIGHT
    top = (axis_top - share * bottom) / (1 - share)
    return (bottom, top), axis_top, brackets


def _bar_axes(n_slots: int) -> tuple[plt.Figure, plt.Axes, tuple[float, float]]:
    """A figure that is exactly its axes, ``STATS_X_UNIT`` wide per slot, and the
    x-range that ends the zero line just past the outer bars."""
    half = STATS_BAR_WIDTH / 2 + STATS_ZERO_MARGIN
    xlim = (-half, n_slots - 1 + half)
    fig = plt.figure(figsize=((xlim[1] - xlim[0]) * STATS_X_UNIT, STATS_HEIGHT))
    return fig, fig.add_axes((0.0, 0.0, 1.0, 1.0)), xlim


def _finish_bar_axes(
    ax: plt.Axes,
    xlim: tuple[float, float],
    ylim: tuple[float, float],
    axis_top: float,
    bracket: float,
    centres: np.ndarray,
    p: float,
    y_axis: bool,
) -> None:
    """The p-value over *centres* at *bracket* (a bracket line when there are two
    centres; over one, shrunk to at most a bar wide), the zero line across the bars,
    and nothing else but the y-axis if *y_axis*, which stops at *axis_top*."""
    # Limits first: an annotation outside them measures as an empty box.
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    if len(centres) > 1:
        ax.plot(centres, [bracket, bracket], color=LINE_COLOR, lw=1.0)
    text = ax.annotate(
        f"p = {p:.3f}",
        (centres.mean(), bracket),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=STATS_P_FONTSIZE,
        color=LINE_COLOR,
    )
    # Text width scales with the font size, so one measurement sets the fit.
    renderer = ax.figure.canvas.get_renderer()
    width = text.get_window_extent(renderer).width / ax.figure.dpi
    bar = STATS_BAR_WIDTH * STATS_X_UNIT
    if len(centres) == 1 and width > bar:
        text.set_fontsize(STATS_P_FONTSIZE * bar / width)
    ax.axhline(0.0, color=LINE_COLOR, lw=STATS_ZERO_WIDTH, zorder=5)
    ax.set_xticks([])
    ax.spines[["top", "right", "bottom"]].set_visible(False)
    if y_axis:
        ax.set_yticks([t for t in ax.get_yticks() if ylim[0] <= t <= axis_top])
        ax.spines["left"].set_bounds(ylim[0], axis_top)
        ax.tick_params(axis="y", labelsize=TICK_FONTSIZE)
    else:
        ax.spines["left"].set_visible(False)
        ax.set_yticks([])


def save_bar(fig: plt.Figure, ax: plt.Axes, stem: Path, y_axis: bool) -> None:
    """Save the axes box, padded by half a tick label above and below so the end tick
    numbers are not cut, widened to the p-value where it overhangs (the one-bar delta
    panels), and with *y_axis* everything left of it too: every method's file has the
    same height, and the same data scale."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    to_inches = fig.dpi_scale_trans.inverted()
    width, height = fig.get_size_inches()
    pad = 0.5 * TICK_FONTSIZE / 72
    text = ax.texts[0].get_window_extent(renderer).transformed(to_inches)
    left, right = min(0.0, text.x0), max(width, text.x1)
    if y_axis:
        left = min(left, ax.get_tightbbox(renderer).transformed(to_inches).x0)
    save(fig, stem, bbox=Bbox([[left, -pad], [right, height + pad]]))


def plot_condition_bars(
    label: str,
    by_condition: dict[str, np.ndarray],
    p: float,
    layout: tuple[tuple[float, float], float, dict[str, float]],
    y_axis: bool,
) -> tuple[plt.Figure, plt.Axes]:
    """One method's Placebo and Psilocybin bars (mean +/- SEM), each participant's
    pair joined, the p-value over a bracket; the baseline's bars hatched."""
    fig, ax, xlim = _bar_axes(len(CONDITIONS))
    xs = np.arange(len(CONDITIONS))
    if STATS_SHOW_PARTICIPANTS:
        for row in np.column_stack([by_condition[c] for c in CONDITIONS]):
            ax.plot(xs, row, color=STATS_PAIR_COLOR, lw=0.8, zorder=2)
            ax.scatter(xs, row, s=10, color=STATS_PAIR_COLOR, zorder=3, linewidths=0)
    for x, c in zip(xs, CONDITIONS):
        mean, sem = _mean_sem(by_condition[c])
        ax.bar(
            x,
            mean,
            width=STATS_BAR_WIDTH,
            color=CONDITION_COLORS[c],
            alpha=0.85,
            hatch=_hatch(label),
            edgecolor="white",
            linewidth=0,
            zorder=1,
        )
        ax.errorbar(x, mean, yerr=sem, color=LINE_COLOR, capsize=4, lw=1.2, zorder=4)
    ylim, axis_top, brackets = layout
    _finish_bar_axes(ax, xlim, ylim, axis_top, brackets[label], xs, p, y_axis)
    return fig, ax


def plot_delta_bar(
    label: str,
    diff: np.ndarray,
    p: float,
    layout: tuple[tuple[float, float], float, dict[str, float]],
    y_axis: bool,
) -> tuple[plt.Figure, plt.Axes]:
    """One method's Psilocybin - Placebo bar (mean +/- SEM), a dot per participant,
    the p-value above; the baseline in its own colour, hatched."""
    fig, ax, xlim = _bar_axes(1)
    mean, sem = _mean_sem(diff)
    baseline = STATS_METHODS[label]["baseline"]
    ax.bar(
        0,
        mean,
        width=STATS_BAR_WIDTH,
        color=STATS_BASELINE_COLOR if baseline else STATS_OURS_COLOR,
        alpha=0.85,
        hatch=_hatch(label),
        edgecolor="white",
        linewidth=0,
        zorder=1,
    )
    ax.errorbar(0, mean, yerr=sem, color=LINE_COLOR, capsize=4, lw=1.2, zorder=4)
    if STATS_SHOW_PARTICIPANTS:
        jitter = np.random.default_rng(0).uniform(-0.12, 0.12, diff.size)
        ax.scatter(jitter, diff, s=10, color=STATS_PAIR_COLOR, zorder=3, linewidths=0)
    ylim, axis_top, brackets = layout
    _finish_bar_axes(ax, xlim, ylim, axis_top, brackets[label], np.zeros(1), p, y_axis)
    return fig, ax

In [ ]:
loaded = {label: method_values(label) for label in STATS_METHODS}
first, second = CONDITIONS
deltas = {label: by_c[second] - by_c[first] for label, (by_c, _) in loaded.items()}
condition_layout = bar_layout(
    {label: _extent(list(by_c.values())) for label, (by_c, _) in loaded.items()}
)
delta_layout = bar_layout({label: _extent([diff]) for label, diff in deltas.items()})
condition_ylim, delta_ylim = condition_layout[0], delta_layout[0]

bars_dir, delta_dir = output_dir("condition_bars"), output_dir("condition_delta")
tag = f"{STATS_SELECTION}__{_GRID_WINDOW}"
for label, (by_condition, test) in loaded.items():
    y_axis = label == STATS_Y_AXIS
    slug = label.lower().replace(" ", "_")
    p = float(test["p"])
    for directory, (fig, ax) in (
        (
            bars_dir,
            plot_condition_bars(label, by_condition, p, condition_layout, y_axis),
        ),
        (delta_dir, plot_delta_bar(label, deltas[label], p, delta_layout, y_axis)),
    ):
        save_bar(fig, ax, directory / f"{slug}__{tag}", y_axis)
        plt.show()
    n = np.isfinite(by_condition[first]).sum()
    print(f"{label}: p = {test['p']:.3f}, n = {n}")

print(
    f"condition bars: {', '.join(CONDITIONS)}, y {condition_ylim[0]:.3g} to "
    f"{condition_ylim[1]:.3g} baseline SD"
)
print(f"delta: {second} − {first}, y {delta_ylim[0]:.3g} to {delta_ylim[1]:.3g}")
print(f"saved to {bars_dir} and {delta_dir}")

## CorrCA CC1 split by the ROI

CC1's shared spatial filter cut in two: its weights on the ASSR ROI electrodes only
(`roi_masked`) and its weights on every other electrode (`non_roi_masked`). The two
parts add up exactly to the full CC1, so they split its response into what the ROI
carries and what the rest of the head carries. Neither part is renormalised; each map
is in units of its own pre-stimulus SD.

- **TF maps** (`tf_grid_corrca_cc1_roi_vs_rest`): the ROI part on top, the rest below;
  Placebo left, Psilocybin right, one scale and one colour bar, as the grids above.
  They are written by the stage-06b grid with `--masked_tf_maps`, so they need one
  rerun: `qsub -v N_PCA=5 jobs/metacentrum/06b-corrca-condition-comparison/run_snr_grid_tracks.pbs`.
- **Topomaps** (`corrca_cc1_filter_roi.pdf`, `corrca_cc1_filter_rest.pdf`): the two
  parts' filter **weights** — the operator each one applies, not a forward pattern —
  drawn by the job's `plot_masked_filter_topomaps`, in `DIVERGING_CMAP` on one shared
  symmetric scale, carried by `corrca_cc1_filter_split_colorbar.pdf` (relative, −1 to
  1). They read the stored fit only, so they need no rerun. They use `corrca_topo` and
  `roi_mask` from the topomap cell above.

Method: [`docs/analyses/06b-corrca-condition-comparison/`](../../docs/analyses/06b-corrca-condition-comparison/README.md), §5.

In [ ]:
SPLIT_STEM = f"tf_grid_corrca_cc{COMPONENT}_roi_vs_rest"
# Row -> (TF store, its key per condition), top to bottom, as in TF_FIGURES.
SPLIT_ROWS = {
    f"CorrCA CC{COMPONENT}, ROI electrodes": (
        CORRCA_TF_PATH,
        "tf_baseline__roi_masked__{}",
    ),
    f"CorrCA CC{COMPONENT}, other electrodes": (
        CORRCA_TF_PATH,
        "tf_baseline__non_roi_masked__{}",
    ),
}
# Operator in corrca_operators -> the signal variant it is and the output stem.
SPLIT_FILTERS = {
    "masked": ("roi_masked", f"corrca_cc{COMPONENT}_filter_roi"),
    "outside": ("non_roi_masked", f"corrca_cc{COMPONENT}_filter_rest"),
}

In [ ]:
stored_keys = set(np.load(CORRCA_TF_PATH).files)
missing = [
    key.format(c)
    for _, key in SPLIT_ROWS.values()
    for c in CONDITIONS
    if key.format(c) not in stored_keys
]
if missing:
    print(
        f"{CORRCA_TF_PATH.name} has no {missing} yet: rerun "
        "run_snr_grid_tracks.pbs (N_PCA=5, named above), then this cell."
    )
else:
    fig, shared, n = plot_tf_grid(SPLIT_ROWS)
    save(fig, tf_dir / SPLIT_STEM)
    plt.show()
    print(f"{SPLIT_STEM}: rows {' / '.join(SPLIT_ROWS)} (top to bottom)")
    print(f"  columns {', '.join(CONDITIONS)}; colour bar ±{shared:.3g} baseline SD")
    print(f"  (n={n}); saved to {tf_dir}")

In [ ]:
split_ops = corrca_operators(
    corrca_topo.extras["corrca_filters"], roi_mask / roi_mask.sum(), roi_mask
)
k = COMPONENT - 1
# One symmetric scale over both parts, so their weights compare directly.
split_limit = max(float(np.abs(split_ops[op][k]).max()) for op in SPLIT_FILTERS)
for operator, (signal, stem) in SPLIT_FILTERS.items():
    kept = roi_mask if operator == "masked" else ~roi_mask
    fig = plot_masked_filter_topomaps(
        split_ops[operator],
        topo_info,
        weight_share=split_ops[f"{operator}_share"],
        n_mask_channels=int(kept.sum()),
        variant=signal,
        comp_indices=[k],
    )
    strip_text(fig)
    image = fig.axes[0].images[0]
    image.set_cmap(DIVERGING_CMAP)
    image.set_clim(-split_limit, split_limit)
    save(fig, topo_dir / stem)
    plt.show()
    share = split_ops[f"{operator}_share"][k]
    print(f"{stem}: {int(kept.sum())} electrodes, {share:.0%} of CC1's |w|; saved")
save_relative_colorbar(
    fig.axes[0], topo_dir / f"corrca_cc{COMPONENT}_filter_split_colorbar"
)
print(f"shared scale: colour bar -1..1 = ±{split_limit:.3g} (filter weight)")